In [ ]:
!pip install music21 tensorflow numpy

In [ ]:
import music21
import numpy as np
import tensorflow as tf

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Bridal-March-(Wedding-March).mid to Bridal-March-(Wedding-March).mid
Saving Dj Sakin& Friends - Turkish Memories.mid to Dj Sakin& Friends - Turkish Memories.mid
Saving Minuet in B flat.mid to Minuet in B flat.mid
Saving Minuet in G.mid to Minuet in G.mid
Saving Canon Fugue n1.mid to Canon Fugue n1.mid
Saving canon-3.mid to canon-3.mid
Saving Beethoven-Moonlight-Sonata.mid to Beethoven-Moonlight-Sonata.mid
Saving Sonata No.14 Op 27 Moonlight Sonata.mid to Sonata No.14 Op 27 Moonlight Sonata.mid
Saving fur-elise.mid to fur-elise.mid
Saving Bagatella Fur Elise.mid to Bagatella Fur Elise.mid


In [ ]:
from music21 import converter

for file in uploaded.keys():
    midi = converter.parse(file)
    print(file, "loaded successfully!")

Bridal-March-(Wedding-March).mid loaded successfully!


MidiException: badly formatted midi bytes, got: b'Rar!\x1a\x07\x00\xcf\x90s\x00\x00\r\x00\x00\x00\x00\x00\x00\x00'

In [ ]:
for file in uploaded.keys():
    try:
        from music21 import converter
        converter.parse(file)
        print("✅", file)
    except Exception as e:
        print("❌", file)

✅ Bridal-March-(Wedding-March).mid
❌ Dj Sakin& Friends - Turkish Memories.mid
✅ Minuet in B flat.mid
✅ Minuet in G.mid
✅ Canon Fugue n1.mid
✅ canon-3.mid
✅ Beethoven-Moonlight-Sonata.mid
✅ Sonata No.14 Op 27 Moonlight Sonata.mid
✅ fur-elise.mid
✅ Bagatella Fur Elise.mid


In [ ]:
from music21 import converter, instrument, note, chord

notes = []

for file in uploaded.keys():
    try:
        midi = converter.parse(file)

        for element in midi.flatten().notes:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))

            elif isinstance(element, chord.Chord):
                notes.append(".".join(str(n) for n in element.normalOrder))

    except:
        print("Skipped:", file)

print("Total notes extracted:", len(notes))
print("First 20 notes:")
print(notes[:20])

Skipped: Dj Sakin& Friends - Turkish Memories.mid
Total notes extracted: 8094
First 20 notes:
['5', '5', 'F2', '5', '5', '5', '5', '5', '5', '5', '5', '10.2.5', 'B-2', '5.10', '5.10', '5.10', 'B-3', 'B-3', '10.2.5', '0.5']


In [ ]:
# Create a list of unique notes
pitchnames = sorted(set(notes))

# Convert each note to a number
note_to_int = {note: number for number, note in enumerate(pitchnames)}

print("Total unique notes:", len(pitchnames))
print("Example mapping:")
print(list(note_to_int.items())[:10])

Total unique notes: 186
Example mapping:
[('0', 0), ('0.1', 1), ('0.2', 2), ('0.2.4', 3), ('0.3', 4), ('0.3.5', 5), ('0.3.7', 6), ('0.4', 7), ('0.4.7', 8), ('0.5', 9)]


In [ ]:
sequence_length = 50

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[n] for n in sequence_in])
    network_output.append(note_to_int[sequence_out])

print("Total training sequences:", len(network_input))

Total training sequences: 8044


In [ ]:
from tensorflow.keras.utils import to_categorical
import numpy as np

# Reshape the input
network_input = np.reshape(
    network_input,
    (len(network_input), sequence_length, 1)
)

# Normalize the values
network_input = network_input / float(len(pitchnames))

# Convert output to categorical
network_output = to_categorical(network_output)

print("Input Shape:", network_input.shape)
print("Output Shape:", network_output.shape)

Input Shape: (8044, 50, 1)
Output Shape: (8044, 186)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(
    256,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(256))

model.add(Dense(256, activation="relu"))

model.add(Dense(len(pitchnames), activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam"
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 186)            │        47,802 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 903,098 (3.45 MB)

 Trainable params: 903,098 (3.45 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    network_input,
    network_output,
    epochs=10,
    batch_size=64
)

Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 79s 598ms/step - loss: 4.2858
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 75s 592ms/step - loss: 4.1870
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 76s 604ms/step - loss: 4.1660
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 76s 604ms/step - loss: 4.1769
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 76s 600ms/step - loss: 4.1773
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 76s 606ms/step - loss: 4.1686
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 75s 596ms/step - loss: 4.1157
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 83s 604ms/step - loss: 4.0932
Epoch 9/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 83s 614ms/step - loss: 4.0003
Epoch 10/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 81s 608ms/step - loss: 3.9746


In [ ]:
import random
import numpy as np

# Create reverse mapping
int_to_note = {number: note for number, note in enumerate(pitchnames)}

# Pick a random starting sequence
start = random.randint(0, len(network_input) - 1)
pattern = network_input[start]

prediction_output = []

# Generate 100 notes
for i in range(100):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)
    result = int_to_note[index]

    prediction_output.append(result)

    # Update the pattern
    pattern = np.append(pattern, index / float(len(pitchnames)))
    pattern = pattern[1:]

print("Generated Notes:")
print(prediction_output[:20])

Generated Notes:
['E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4']


In [ ]:
from music21 import stream, note, chord

offset = 0
output_notes = []

for pattern in prediction_output:

    if '.' in pattern or pattern.isdigit():
        notes_in_chord = pattern.split('.')
        chord_notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.offset = offset
            chord_notes.append(new_note)

        new_chord = chord.Chord(chord_notes)
        output_notes.append(new_chord)

    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)

midi_stream.write('midi', fp='generated_music.mid')

print("✅ Music generated successfully!")

✅ Music generated successfully!


In [1]:
from google.colab import files

files.download("generated_music.mid")

FileNotFoundError: Cannot find file: generated_music.mid

In [3]:
import os

print(os.listdir())

['.config', 'sample_data']


In [4]:
from music21 import stream, note, chord

offset = 0
output_notes = []

for pattern in prediction_output:

    if "." in pattern:
        notes_in_chord = pattern.split(".")
        chord_notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.offset = offset
            chord_notes.append(new_note)

        output_notes.append(chord.Chord(chord_notes))

    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("✅ generated_music.mid created")

NameError: name 'prediction_output' is not defined

In [5]:
print(notes[:5])

NameError: name 'notes' is not defined

In [8]:
import music21
import numpy as np
import tensorflow as tf

In [9]:
from google.colab import files

uploaded = files.upload()

Saving Bridal-March-(Wedding-March).mid to Bridal-March-(Wedding-March).mid
Saving Dj Sakin& Friends - Turkish Memories.mid to Dj Sakin& Friends - Turkish Memories.mid
Saving Minuet in B flat.mid to Minuet in B flat.mid
Saving Minuet in G.mid to Minuet in G.mid
Saving Canon Fugue n1.mid to Canon Fugue n1.mid
Saving canon-3.mid to canon-3.mid
Saving Beethoven-Moonlight-Sonata.mid to Beethoven-Moonlight-Sonata.mid
Saving Sonata No.14 Op 27 Moonlight Sonata.mid to Sonata No.14 Op 27 Moonlight Sonata.mid
Saving fur-elise.mid to fur-elise.mid
Saving Bagatella Fur Elise.mid to Bagatella Fur Elise.mid


In [10]:
from music21 import converter, note, chord

notes = []

for file in uploaded.keys():
    try:
        midi = converter.parse(file)

        for element in midi.flatten().notes:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append(".".join(str(n) for n in element.normalOrder))

    except:
        print("Skipped:", file)

print("Total notes extracted:", len(notes))

Skipped: Dj Sakin& Friends - Turkish Memories.mid
Total notes extracted: 8094


In [11]:
pitchnames = sorted(set(notes))

note_to_int = {note: number for number, note in enumerate(pitchnames)}

print("Total unique notes:", len(pitchnames))

Total unique notes: 186


In [12]:
sequence_length = 50

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[n] for n in sequence_in])
    network_output.append(note_to_int[sequence_out])

print("Total training sequences:", len(network_input))


Total training sequences: 8044


In [13]:
from tensorflow.keras.utils import to_categorical
import numpy as np

# Reshape the input
network_input = np.reshape(
    network_input,
    (len(network_input), sequence_length, 1)
)

# Normalize the input
network_input = network_input / float(len(pitchnames))

# Convert output to categorical
network_output = to_categorical(network_output)

print("Input Shape:", network_input.shape)
print("Output Shape:", network_output.shape)

Input Shape: (8044, 50, 1)
Output Shape: (8044, 186)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(
    256,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(256))

model.add(Dense(256, activation="relu"))

model.add(Dense(len(pitchnames), activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam"
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 186)            │        47,802 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 903,098 (3.45 MB)

 Trainable params: 903,098 (3.45 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(
    network_input,
    network_output,
    epochs=10,
    batch_size=64
)

Epoch 1/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 65s 481ms/step - loss: 4.2731
Epoch 2/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 84s 498ms/step - loss: 4.1864
Epoch 3/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 62s 491ms/step - loss: 4.1834
Epoch 4/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 60s 480ms/step - loss: 4.1785
Epoch 5/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 62s 490ms/step - loss: 4.1760
Epoch 6/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 82s 488ms/step - loss: 4.1696
Epoch 7/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 82s 490ms/step - loss: 4.1602
Epoch 8/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 62s 491ms/step - loss: 4.1170
Epoch 9/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 82s 492ms/step - loss: 4.0301
Epoch 10/10
126/126 ━━━━━━━━━━━━━━━━━━━━ 62s 491ms/step - loss: 3.9859


In [16]:
import random
import numpy as np

# Create reverse mapping
int_to_note = {number: note for number, note in enumerate(pitchnames)}

# Pick a random starting sequence
start = random.randint(0, len(network_input) - 1)
pattern = network_input[start]

prediction_output = []

# Generate 100 notes
for i in range(100):
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)
    result = int_to_note[index]

    prediction_output.append(result)

    pattern = np.append(pattern, index / float(len(pitchnames)))
    pattern = pattern[1:]

print("Generated Notes:")
print(prediction_output[:20])

Generated Notes:
['E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4', 'E4']


In [17]:
from music21 import stream, note, chord

offset = 0
output_notes = []

for pattern in prediction_output:

    if "." in pattern:
        notes_in_chord = pattern.split(".")
        chord_notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.offset = offset
            chord_notes.append(new_note)

        output_notes.append(chord.Chord(chord_notes))

    else:
        new_note = note.Note(pattern)
        new_note.offset = offset
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("✅ MIDI file created successfully!")

✅ MIDI file created successfully!


In [19]:
from google.colab import files

files.download("generated_music.mid")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
model.save("music_model.keras")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [21]:
from google.colab import files

files.download("music_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>